# Task 1: Sales & Demand Forecasting
## Future Interns - Machine Learning Internship
**CIN:** FIT/JUN26/ML9423  
**Author:** Abdi Negash  
**Date:** June 2026

### Project Goal
Build a model to forecast future sales using historical business data.

## Section 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

warnings.filterwarnings("ignore")

os.makedirs("../images", exist_ok=True)
os.makedirs("../models", exist_ok=True)
os.makedirs("../reports", exist_ok=True)

print("Libraries imported successfully!")

## Section 2: Load Dataset

**Why:** We load the historical sales data to understand its structure before processing.

In [ ]:
df = pd.read_csv("../data/sales.csv", sep="	")

print(f"Dataset Shape: {df.shape}")
print(f"
Columns: {df.columns.tolist()}")
print(f"
First 5 Rows:")
df.head()

## Section 3: Data Inspection

**Why:** Understanding data types, missing values, and statistics helps us plan cleaning steps.

In [ ]:
print("Data Types:")
print(df.dtypes)

print("
Missing Values:")
print(df.isnull().sum())

print("
Basic Statistics:")
df.describe()

## Section 4: Data Cleaning

**Why:** Dirty data causes poor predictions. We remove duplicates and fix formats.

In [ ]:
missing_before = df.isnull().sum().sum()
print(f"Missing values before cleaning: {missing_before}")

duplicates_before = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Duplicate rows removed: {duplicates_before}")

df["date"] = pd.to_datetime(df["date"])
print(f"Date column converted to datetime format")

missing_after = df.isnull().sum().sum()
print(f"Missing values after cleaning: {missing_after}")
print("
Data Cleaning Complete!")

## Section 5: Feature Engineering

**Why:** ML models cannot understand dates directly. We extract numerical features.

In [ ]:
df["Year"] = df["date"].dt.year
df["Month"] = df["date"].dt.month
df["Day"] = df["date"].dt.day
df["DayOfWeek"] = df["date"].dt.dayofweek
df["DayOfYear"] = df["date"].dt.dayofyear

print("New Features Created: Year, Month, Day, DayOfWeek, DayOfYear")
print("
Sample after Feature Engineering:")
df[["date", "Year", "Month", "Day", "DayOfWeek", "sales"]].head()

## Section 6: Exploratory Data Analysis (EDA)

**Why:** EDA helps us understand patterns before modeling.

In [ ]:
STORE_NBR = 1
FAMILY = "GROCERY I"

df_filtered = df[(df["store_nbr"] == STORE_NBR) & (df["family"] == FAMILY)].copy()
df_filtered = df_filtered.sort_values("date")

print(f"Filtered data: {len(df_filtered)} rows for Store {STORE_NBR}, {FAMILY}")

plt.figure(figsize=(14, 5))
plt.plot(df_filtered["date"], df_filtered["sales"], linewidth=0.8, color="steelblue")
plt.title(f"Sales Trend Over Time - Store {STORE_NBR}, {FAMILY}", fontsize=14, fontweight="bold")
plt.xlabel("Date")
plt.ylabel("Sales ($)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../images/sales_trend.png", dpi=150)
plt.show()
print("Saved: ../images/sales_trend.png")

In [ ]:
monthly_sales = df_filtered.groupby("Month")["sales"].mean()

plt.figure(figsize=(10, 5))
monthly_sales.plot(kind="bar", color="coral", edgecolor="black")
plt.title(f"Average Monthly Sales - Store {STORE_NBR}, {FAMILY}", fontsize=14, fontweight="bold")
plt.xlabel("Month")
plt.ylabel("Average Sales ($)")
plt.xticks(rotation=0)
plt.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("../images/monthly_sales.png", dpi=150)
plt.show()
print("Saved: ../images/monthly_sales.png")

## Section 7: Prepare Data for Machine Learning

In [ ]:
features = ["Year", "Month", "Day", "DayOfWeek", "DayOfYear", "onpromotion"]
X = df_filtered[features]
y = df_filtered["sales"]

print(f"Features (X): {features}")
print(f"Target (y): sales")
print(f"
Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")

## Section 8: Train-Test Split

**Why:** 80% training, 20% testing for fair evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=False
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")

## Section 9: Linear Regression Model

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_predictions = lr_model.predict(X_test)

print("Linear Regression trained successfully!")

## Section 10: Random Forest Model

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)

print("Random Forest trained successfully!")

## Section 11: Model Evaluation

**Metrics:**
- MAE = Mean Absolute Error (average mistake in dollars)
- RMSE = Root Mean Squared Error (penalizes big mistakes)
- R² = R-squared (1.0 = perfect prediction)

In [ ]:
def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

lr_mae, lr_rmse, lr_r2 = evaluate(y_test, lr_predictions)
rf_mae, rf_rmse, rf_r2 = evaluate(y_test, rf_predictions)

print("=" * 50)
print(f"{"Metric":<20} {"Linear Regression":<18} {"Random Forest":<18}")
print("=" * 50)
print(f"{"MAE":<20} {lr_mae:<18.4f} {rf_mae:<18.4f}")
print(f"{"RMSE":<20} {lr_rmse:<18.4f} {rf_rmse:<18.4f}")
print(f"{"R2 Score":<20} {lr_r2:<18.4f} {rf_r2:<18.4f}")
print("=" * 50)

best_model = rf_model if rf_r2 > lr_r2 else lr_model
best_name = "Random Forest" if rf_r2 > lr_r2 else "Linear Regression"
print(f"
Best Model: {best_name} (higher R2 score)")

## Section 12: Actual vs Predicted Visualization

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(y_test, rf_predictions, alpha=0.5, color="darkgreen", s=20)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", linewidth=2, label="Perfect Prediction")
plt.xlabel("Actual Sales ($)")
plt.ylabel("Predicted Sales ($)")
plt.title(f"Actual vs Predicted Sales - {best_name}", fontsize=14, fontweight="bold")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../images/actual_vs_predicted.png", dpi=150)
plt.show()
print("Saved: ../images/actual_vs_predicted.png")

## Section 13: Future Sales Forecasting

In [ ]:
from datetime import datetime, timedelta

future_dates = pd.DataFrame({
    "date": pd.date_range(start="2017-07-01", periods=90, freq="D")
})
future_dates["Year"] = future_dates["date"].dt.year
future_dates["Month"] = future_dates["date"].dt.month
future_dates["Day"] = future_dates["date"].dt.day
future_dates["DayOfWeek"] = future_dates["date"].dt.dayofweek
future_dates["DayOfYear"] = future_dates["date"].dt.dayofyear
future_dates["onpromotion"] = 0

future_X = future_dates[features]
future_predictions = best_model.predict(future_X)

plt.figure(figsize=(12, 5))
plt.plot(future_dates["date"], future_predictions, color="purple", linewidth=2, marker="o", markersize=3)
plt.title("Future Sales Forecast (July - September 2017)", fontsize=14, fontweight="bold")
plt.xlabel("Date")
plt.ylabel("Predicted Sales ($)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../images/future_forecast.png", dpi=150)
plt.show()
print("Saved: ../images/future_forecast.png")

forecast_df = pd.DataFrame({
    "Date": future_dates["date"],
    "Predicted_Sales": future_predictions
})
forecast_df.to_csv("../reports/future_sales_forecast.csv", index=False)
print("Saved: ../reports/future_sales_forecast.csv")

## Section 14: Feature Importance

In [ ]:
importances = rf_model.feature_importances_
feature_names = features

plt.figure(figsize=(10, 6))
indices = np.argsort(importances)[::-1]
plt.bar(range(len(importances)), importances[indices], color="teal", edgecolor="black")
plt.xticks(range(len(importances)), [feature_names[i] for i in indices], rotation=45)
plt.title("Feature Importance - Random Forest", fontsize=14, fontweight="bold")
plt.ylabel("Importance")
plt.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("../images/feature_importance.png", dpi=150)
plt.show()
print("Saved: ../images/feature_importance.png")

for i in indices:
    print(f"{feature_names[i]}: {importances[i]*100:.2f}%")

## Section 15: Save Best Model

In [ ]:
joblib.dump(best_model, "../models/sales_forecast_model.pkl")
print(f"Model saved: ../models/sales_forecast_model.pkl")
print(f"Model type: {best_name}")

## Section 16: Business Insights & Conclusion

### Project Summary
This project predicts future sales using historical store data. The workflow included data loading, cleaning, feature engineering, model training, evaluation, and forecasting.

### Model Comparison Results
- **Linear Regression:** MAE=393.85, RMSE=503.06, R²=0.2562
- **Random Forest:** MAE=217.70, RMSE=336.56, R²=0.6671
- **Winner:** Random Forest Regressor

### Key Business Insights
1. **Day of week** is the strongest sales predictor (58.6% importance)
2. **Seasonality** significantly impacts sales patterns
3. Random Forest captures non-linear relationships better than Linear Regression
4. Future forecasts enable better inventory and staffing decisions
5. Promotions (onpromotion feature) have minimal impact on this dataset

### Possible Improvements
1. Include all stores and product families for better generalization
2. Add lag features (sales from previous days)
3. Try XGBoost or Prophet for time series forecasting
4. Incorporate external factors (holidays, oil prices, weather)
5. Deploy as a web API or dashboard for real-time use